In [1]:
from datetime import datetime, timedelta
from typing import Iterable, Optional, Set, Dict, List,  Sequence,  Tuple
import pandas as pd
from zoneinfo import ZoneInfo
import numpy as np
import logging
import os
import re
import math

from sqlalchemy import create_engine, text, inspect
from sqlalchemy.engine import Engine, Connection
from sqlalchemy.exc import SQLAlchemyError
import pymysql
from concurrent.futures import ThreadPoolExecutor, as_completed
# -*- coding: utf-8 -*-

import warnings
warnings.filterwarnings('ignore')

In [2]:
class MySQLConnet:
    def __init__(self, dbname):
        self.db = dbname
        host = "10.97.142.217"
        username = "l6a01_user"
        password = "l6a01$user"
        self.engine = create_engine(f"mysql+pymysql://{username}:{password}@{host}/{dbname}")

    def list_tables(self):
        try:
            inspector = inspect(self.engine)
            tables = inspector.get_table_names()
            logging.info(f"[list_tables] 成功取得資料表名稱，共 {len(tables)} 張表。")
            return tables
        except SQLAlchemyError as e:
            logging.error(f"[list_tables] 取得資料表名稱時發生錯誤: {e}")
            return []
    
    def get_table(self, table_name):
        try:
            df = pd.read_sql_table(table_name, self.engine)
            #logging.info(f"get_table- 讀取資料表 '{table_name}' 成功 ({len(df)} rows).")
            return df
        except SQLAlchemyError as e:
            logging.error(f"[get_table] 讀取 '{table_name}' 發生錯誤: {e}")
            return pd.DataFrame()
        
   
    # ---------- 給 /api/defect_data 用：明細 ----------
    def get_defects_by_key(self, table: str, key_dict: dict):
        """
        key_dict: {"gid": glass_id, "rid": recipe_id, "t": scantime('YYYY-MM-DD HH:MM:SS')}
        會把 x,y 轉成數值欄位；欄位命名對齊前端（size/img/chip/type）。
        """
        sql = f"""
            SELECT
                CAST(x AS UNSIGNED)       AS x,
                CAST(y AS UNSIGNED)       AS y,
                defect_size               AS size,
                pic_name                  AS img,
                chip_name                 AS chip
            FROM `{table}`
            WHERE glass_id = :gid
              AND recipe_id = :rid
              AND scantime = :t
        """
        try:
            with self.engine.begin() as conn:
                rows = conn.execute(text(sql), key_dict).mappings().all()
                # 轉成 list[dict]，避免 RowMapping 不能 JSON 的問題
                return [dict(r) for r in rows]
        except SQLAlchemyError as e:
            logging.error(f"[get_defects_by_key] {e}")
            return []
    # --- 小工具：檢查 & 引用識別字 ---
    _IDENT_RE = re.compile(r"^[A-Za-z0-9_]+$")

    def _validate_ident(self, name: str) -> str:
        if not isinstance(name, str) or not self._IDENT_RE.match(name):
            raise ValueError(f"非法識別字: {name!r}")
        return name

    def _qual_table(self, table_name: str) -> str:
        self._validate_ident(table_name)
        return f"`{self.db}`.`{table_name}`"

    def table_exists(self, table_name: str) -> bool:
        insp = inspect(self.engine)
        return insp.has_table(table_name, schema=self.db)

    # 1) 儲存資料表：fillna('')、去重、存在覆蓋/不存在建立
    def save_table(self, table_name: str, df: pd.DataFrame, chunksize: int = 10000) -> int:
        """
        將 DataFrame 儲存為 {db}.{table_name}：
          - 先把「文字型欄位」的 NaN -> ''（避免把數值欄轉成字串）
          - 去除重複列
          - if_exists='replace'：存在則覆蓋，不在則新建
        回傳：實際寫入列數
        """
        if df is None:
            raise ValueError("df 不能是 None")
        if not isinstance(df, pd.DataFrame):
            raise TypeError("df 需為 pandas.DataFrame")

        df = df.copy()

        # 僅替換「文字欄位」的 NaN，避免數值欄被轉字串
        obj_cols = df.select_dtypes(include=["object"]).columns
        if len(obj_cols) > 0:
            df[obj_cols] = df[obj_cols].fillna('')

        # 去重複
        before = len(df)
        df.drop_duplicates(inplace=True)
        after = len(df)

        # 寫入（覆蓋）
        # 空表時 to_sql 也能建立 schema（依據 df 的欄型）；若完全空 df 無法推斷型別，請傳入 dtype 參數。
        df.to_sql(
            name=table_name,
            con=self.engine,
            schema=self.db,
            if_exists='replace',
            index=False,
            chunksize=chunksize,
            method='multi'
        )

        logging.info(f"[save_table] {self.db}.{table_name} 已寫入 {after} 列（去除 {before-after} 重複）")
        return after

    def _columns_and_types(self, table_name: str):
        """
        回傳 [(column_name, data_type), ...]
        以明確別名 col/typ 取值，避免大小寫與驅動差異造成的 NoSuchColumnError。
        """
        sql = text("""
            SELECT
                COLUMN_NAME AS col,
                DATA_TYPE   AS typ
            FROM information_schema.columns
            WHERE table_schema = :db AND table_name = :tbl
            ORDER BY ORDINAL_POSITION
        """)
        with self.engine.begin() as conn:
            rp = conn.execute(sql, {"db": self.db, "tbl": table_name})
            try:
                rows = rp.mappings().all()
                return [(r["col"], r["typ"]) for r in rows]
            except Exception:
                # 某些驅動可能不支援 mappings；退回以位置索引取值
                rows = rp.fetchall()
                return [(r[0], r[1]) for r in rows]

    def append_or_create_dedup(
        self,
        table_name: str,
        df: pd.DataFrame,
        dedup_keys: list[str] | None = None,
        *,
        text_na: str = "nan",
        chunksize: int = 10_000
    ) -> int:
        """
        若表不存在 → 直接建立並寫入 df（先去重、文字欄位空值補 'nan'）。
        若表存在 → 將 df 寫入暫存表，再以 NOT EXISTS 去重後插入正式表。
        之後把正式表「文字欄位」中的 NULL 一次性補成 'nan'（避免殘留空值）。
        備註：
          - 去重鍵 `dedup_keys` 未指定時，採用「df 與目標表的共通欄位」作為比對鍵（全欄位完全一致才視為重複）。
          - 數值欄位保留 NULL（不以字串補值，避免型別污染）。
        回傳：實際新增列數（不含跳過的重複列）。
        """
        if df is None or not isinstance(df, pd.DataFrame):
            raise ValueError("df 必須是 pandas.DataFrame")

        df = df.copy().reset_index(drop=True)

        # 文字/字串/分類欄位的空值補 'nan'（數值欄位不處理，避免轉型）
        obj_like = list(df.select_dtypes(include=["object", "string", "category"]).columns)
        if obj_like:
            df[obj_like] = df[obj_like].astype("string").fillna(text_na)

        # 先行去重（減少寫入量）
        before = len(df)
        df.drop_duplicates(inplace=True, ignore_index=True)
        after = len(df)
        dropped_local = before - after

        tbl_qual = self._qual_table(table_name)

        with self.engine.begin() as conn:
            if not self.table_exists(table_name):
                # 表不存在：直接建立
                df.to_sql(
                    name=table_name,
                    con=self.engine,
                    schema=self.db,
                    if_exists="fail",
                    index=False,
                    chunksize=chunksize,
                    method="multi",
                )
                # 將文字欄位的 NULL（若有）統一補 'nan'
                cols_types = self._columns_and_types(table_name)
                char_types = {"char", "varchar", "tinytext", "text", "mediumtext", "longtext"}
                text_cols = [c for c, t in cols_types if t.lower() in char_types]
                for c in text_cols:
                    conn.execute(text(f"UPDATE {tbl_qual} SET `{c}` = :na WHERE `{c}` IS NULL"), {"na": text_na})
                logging.info(
                    f"[append_or_create_dedup] 建立新表 {self.db}.{table_name} 並寫入 {after} 列（df 端先去除 {dropped_local} 重複）。"
                )
                return after

            # 表已存在：寫入暫存表後去重插入
            # 取得目標表欄位與型別
            cols_types = self._columns_and_types(table_name)
            target_cols = [c for c, _ in cols_types]

            # 只保留 df 中存在於目標表的欄位（避免 schema 不一致）
            use_cols = [c for c in df.columns if c in target_cols]
            if not use_cols:
                logging.warning("[append_or_create_dedup] df 欄位與目標表無交集，無法寫入。")
                return 0
            df_use = df[use_cols].copy()

            # 暫存表名稱
            stg_name = f"__stg_{table_name}_{int(datetime.now().timestamp())}"
            stg_qual = self._qual_table(stg_name)

            # 建立暫存表（replace 可確保不存在）
            df_use.to_sql(
                name=stg_name,
                con=self.engine,
                schema=self.db,
                if_exists="replace",
                index=False,
                chunksize=chunksize,
                method="multi",
            )

            # 去重鍵（未指定 → 用所有共通欄位）
            if dedup_keys:
                # 僅保留鍵中存在於目標表的欄位
                keys = [k for k in dedup_keys if k in use_cols]
                if not keys:
                    logging.warning("[append_or_create_dedup] dedup_keys 不在目標表中，改用全欄位去重。")
                    keys = use_cols
            else:
                keys = use_cols

            # 準備 INSERT ... SELECT NOT EXISTS（使用 NULL-safe 相等 `<=>`）
            col_list = ", ".join(f"`{c}`" for c in use_cols)
            sel_list = ", ".join(f"s.`{c}`" for c in use_cols)
            cond = " AND ".join(f"(t.`{k}` <=> s.`{k}`)" for k in keys)

            insert_sql = f"""
                INSERT INTO {tbl_qual} ({col_list})
                SELECT {sel_list}
                FROM {stg_qual} AS s
                WHERE NOT EXISTS (
                    SELECT 1 FROM {tbl_qual} AS t
                    WHERE {cond}
                )
            """
            res = conn.execute(text(insert_sql))
            inserted = res.rowcount or 0

            # 刪除暫存表
            conn.execute(text(f"DROP TABLE IF EXISTS {stg_qual}"))

            # 文字欄位 NULL → 'nan'（僅更新為 NULL 的）
            char_types = {"char", "varchar", "tinytext", "text", "mediumtext", "longtext"}
            text_cols = [c for c, t in cols_types if t.lower() in char_types]
            for c in text_cols:
                conn.execute(text(f"UPDATE {tbl_qual} SET `{c}` = :na WHERE `{c}` IS NULL"), {"na": text_na})

            logging.info(
                f"[append_or_create_dedup] 追加完成：插入 {inserted} 列；df 端先去除 {dropped_local} 重複。"
            )
            return inserted

dbhandler = MySQLConnet("l6a01_project")

In [3]:
tbns = dbhandler.list_tables()
tbns

['aoi100_capa_hourly_rawdata',
 'aoi100_capa_summary',
 'aoi100_glassdata_202511',
 'aoi100_glassdata_202512',
 'aoi100_pidensity_202509_pi000',
 'aoi100_pidensity_202509_pi100',
 'aoi100_pidensity_202509_pi200',
 'aoi100_pidensity_202509_pi300',
 'aoi100_pidensity_202509_pi400',
 'aoi100_pidensity_202509_pi500',
 'aoi100_pidensity_202509_pi600',
 'aoi100_pidensity_202509_pi700',
 'aoi100_pidensity_202510_pi000',
 'aoi100_pidensity_202510_pi100',
 'aoi100_pidensity_202510_pi200',
 'aoi100_pidensity_202510_pi300',
 'aoi100_pidensity_202510_pi400',
 'aoi100_pidensity_202510_pi500',
 'aoi100_pidensity_202510_pi600',
 'aoi100_pidensity_202510_pi700',
 'aoi100_pidensity_202511_pi000',
 'aoi100_pidensity_202511_pi100',
 'aoi100_pidensity_202511_pi200',
 'aoi100_pidensity_202511_pi300',
 'aoi100_pidensity_202511_pi400',
 'aoi100_pidensity_202511_pi500',
 'aoi100_pidensity_202511_pi600',
 'aoi100_pidensity_202511_pi700',
 'aoi100_pidensity_202512_pi000',
 'aoi100_pidensity_202512_pi100',
 'aoi

In [22]:
data = {}
for tbn in [v for v in tbns if 'pidenisty_pihour_' in v]:
    tb = dbhandler.get_table(tbn)
    print(tbn, len(tb))
    print(tb.iloc[0,:].to_dict())
    data[tbn] = tb
    
    """
    if 'noteText' in tb.columns:
        tb = tb.rename(columns={'noteText': 'comment'}, inplace=False)
        dbhandler.save_table(tbn, tb)
    """
        

pidenisty_pihour_202511 3608
{'pi_hour': Timestamp('2025-11-03 07:00:00'), 'aoi': 'aoi200', 'line_id': 'CAPIC500', 'model': 'M270DAN8S', 'glass_type': 'TFT', 'recipe_id': '0286', 'ai_code_1': 'Not_Found', 'maingroup_glass_count': 4, 'maingroup_defect_count': 421, 'defect_code_glass_count': 4, 'defect_code_count': 17, 'small_defect_count': 8, 'middle_defect_count': 9, 'large_defect_count': 0, 'over_defect_count': 0, 'glass': 'VQ5AANS8C,VQ5AANS8D,VQ5AANS8E,VQ5AANS8F', 'pic_paths': 'http://10.97.139.98:1454/CAAOI202/CA007001/JF5AATC8C/PCS1/20251104051010/', 'comment': ''}
pidenisty_pihour_202512 2438
{'pi_hour': Timestamp('2025-12-01 22:00:00'), 'aoi': 'aoi100', 'line_id': 'CAPIC300', 'model': 'B140UAN08', 'glass_type': 'CF', 'recipe_id': '202', 'ai_code_1': 'BM_Cover', 'maingroup_glass_count': 17, 'maingroup_defect_count': 1171, 'defect_code_glass_count': 2, 'defect_code_count': 2, 'small_defect_count': 2, 'middle_defect_count': 0, 'large_defect_count': 0, 'over_defect_count': 0, 'glass'

In [ ]:
dbhandler.

In [11]:
new_tbn = 'aoi_density_spec_for_aoimonitor'
tbn = 'aoi_spec_for_aoimonitor'
spec_df= dbhandler.get_table(tbn)

#
spec_df.drop_duplicates(subset=spec_df.columns, keep='first', inplace=True)
print(len(spec_df))
#dbhandler.save_table(tbn, spec_df2.iloc[:,1:])
spec_df.iloc[0,:].to_dict()

1060


{'NO': '1',
 'MODEL_ID': 'B070AAN01',
 'MODEL_TYPE': 'Normal',
 'DEFECT_CODE': 'PI_Spot_NP',
 'PROCESS_TYPE': 'BIG',
 'SIZE_TYPE': 'OL',
 'OOC': '0.33',
 'OOS': '0.57'}

In [13]:
for cl, val in zip(['Editor', 'modify_time', 'drop'],['預設','2025-12-09 08:00:00', 'F']):
    spec_df[cl] = val

dbhandler.save_table(new_tbn, spec_df.iloc[:,1:])

1060

In [15]:
spec_df.iloc[:,1:].columns

Index(['MODEL_ID', 'MODEL_TYPE', 'DEFECT_CODE', 'PROCESS_TYPE', 'SIZE_TYPE',
       'OOC', 'OOS', 'Editor', 'modify_time', 'drop'],
      dtype='object')

In [ ]:

print(len(spec_df))
#dbhandler.save_table('aoi_spec_for_aoimonitor', spec_df.iloc[:,:-2])
spec_df.head()

1060


,NO,MODEL_ID,MODEL_TYPE,DEFECT_CODE,PROCESS_TYPE,SIZE_TYPE,OOC,OOS,Editor,modify_time
0,1,B070AAN01,Normal,PI_Spot_NP,BIG,OL,0.33,0.57,預設,2025-12-09 08:00:00
1,1,B070AAN01,Normal,PI_Spot_NP,BIG,OLMS,300,300,預設,2025-12-09 08:00:00
2,1,B070AAN01,Normal,PIS With Particle,BIG,OL,0.28,0.33,預設,2025-12-09 08:00:00
3,1,B070AAN01,Normal,PIS With Particle,BIG,OLMS,2,3,預設,2025-12-09 08:00:00
4,1,B070AAN01,Normal,Polymer,BIG,OLMS,5,10,預設,2025-12-09 08:00:00


In [9]:
now = datetime.now()
strtime = now.strftime("%Y-%m-%d %H:%M:%S")
strtime

'2025-12-09 08:44:30'

In [ ]:
default_spec_coldict = {'NO': 'NO',
                        'MODEL_ID': 'model',
                        'MODEL_TYPE': 'MODEL_TYPE',
                        'DEFECT_CODE': 'ai_code_1',
                        'PROCESS_TYPE': 'PROCESS_TYPE',
                        'SIZE_TYPE': 'SIZE_TYPE',
                        'OOC': 'OOC',
                        'OOS': 'OOS'}
spec_df = spec_df.rename(columns={val:key for key, val in default_spec_coldict.items()}, inplace=False).fillna('')

spec_df.head()

,NO,MODEL_ID,MODEL_TYPE,DEFECT_CODE,PROCESS_TYPE,SIZE_TYPE,OOC,OOS,Editor,modify_time
0,1,B070AAN01,Normal,PI_Spot_NP,BIG,OL,0.33,0.57,預設,2025-12-09 08:00:00
1,1,B070AAN01,Normal,PI_Spot_NP,BIG,OLMS,300,300,預設,2025-12-09 08:00:00
2,1,B070AAN01,Normal,PIS With Particle,BIG,OL,0.28,0.33,預設,2025-12-09 08:00:00
3,1,B070AAN01,Normal,PIS With Particle,BIG,OLMS,2,3,預設,2025-12-09 08:00:00
4,1,B070AAN01,Normal,Polymer,BIG,OLMS,5,10,預設,2025-12-09 08:00:00


In [27]:
#spec_df['Editor'] = '預設'
#spec_df['modify_time'] = '2025-12-09 08:00:00'
dbhandler.save_table(tbn, spec_df)

2120

In [7]:
spec_df['SIZE_TYPE'].unique()

array(['OL', 'OLMS'], dtype=object)

In [5]:
spec_df[spec_df['MODEL_ID'] == 'B160UAN4A']

,NO,MODEL_ID,MODEL_TYPE,DEFECT_CODE,PROCESS_TYPE,SIZE_TYPE,OOC,OOS
568,25,B160UAN4A,Normal,PI_Spot_NP,BIG,OL,0.33,0.57
569,25,B160UAN4A,Normal,PI_Spot_NP,BIG,OLMS,300,300
570,25,B160UAN4A,Normal,PIS With Particle,BIG,OL,0.28,0.33
571,25,B160UAN4A,Normal,PIS With Particle,BIG,OLMS,2,3
572,25,B160UAN4A,Normal,Polymer,BIG,OLMS,5,10
573,25,B160UAN4A,Normal,SSIU_Polymer,BIG,OLMS,10,15


In [26]:
def rename_columns_by_dict(df: pd.DataFrame, mapping: dict):
    """
    依照 mapping(dict) 替換 DataFrame 欄位名稱。
    mapping 的 key 是舊欄位名，value 是新欄位名。
    回傳替換後的新 DataFrame（不修改原 df）。
    """
    if not isinstance(mapping, dict):
        raise ValueError("mapping 必須是 dict，格式如 {'old_name': 'new_name'}")

    # pandas rename
    new_df = df.rename(columns=mapping, inplace=False)

    return new_df


In [31]:
spec_df.columns

Index(['NO', 'MODEL_ID', 'MODEL_TYPE', 'DEFECT_CODE', 'PROCESS_TYPE',
       'SIZE_TYPE', 'OOC', 'OOS'],
      dtype='object')

In [35]:
coldict = {'NO': 'NO',
 'MODEL_ID': 'model',
 'MODEL_TYPE': 'MODEL_TYPE',
 'DEFECT_CODE': 'ai_code_1',
 'PROCESS_TYPE': 'PROCESS_TYPE',
 'SIZE_TYPE': 'SIZE_TYPE',
 'OOC': 'OOC',
 'OOS': 'OOS'}

spec_df2 = rename_columns_by_dict(spec_df, coldict)
"""
spec_df.columns = ['NO', 'MODEL_ID', 'MODEL_TYPE', 'DEFECT_CODE', 'PROCESS_TYPE',
       'SIZE_TYPE', 'OOC', 'OOS']
"""
spec_df2.iloc[0,:].to_dict()

{'NO': '1',
 'model': 'B070AAN01',
 'MODEL_TYPE': 'Normal',
 'ai_code_1': 'PI_Spot_NP',
 'PROCESS_TYPE': 'BIG',
 'SIZE_TYPE': 'OL',
 'OOC': '0.33',
 'OOS': '0.57'}

In [16]:
spec_df['SIZE_TYPE'].unique()

array(['OL', 'OLMS'], dtype=object)

In [39]:
def fetch_latest_spec_summary(mysql: MySQLConnet,
                              table_name: str = "aoi_density_fixed_spec_table") -> pd.DataFrame:
    """
    從 his_90days_spec_table 撈取：
    依 ('line_id','aoi','model','recipe_id','glass_type','ai_code_1','size_key') 分群，
    取 GEN_DT 最大 (最新) 的那一筆資料。
    """

    engine = mysql.engine
    dbname = mysql.db

    sql = f"""
        SELECT t.*
        FROM `{dbname}`.`{table_name}` AS t
        INNER JOIN (
            SELECT
                line_id, aoi, model, recipe_id, glass_type, ai_code_1, size_key,
                MAX(GEN_DT) AS max_dt
            FROM `{dbname}`.`{table_name}`
            GROUP BY
                line_id, aoi, model, recipe_id, glass_type, ai_code_1, size_key
        ) AS g
        ON  t.line_id     = g.line_id
        AND t.aoi         = g.aoi
        AND t.model       = g.model
        AND t.recipe_id   = g.recipe_id
        AND t.glass_type  = g.glass_type
        AND t.ai_code_1   = g.ai_code_1
        AND t.size_key    = g.size_key
        AND t.GEN_DT      = g.max_dt
    """

    with engine.begin() as conn:
        df = pd.read_sql_query(sql, conn)

    return df

df = fetch_latest_spec_summary(MySQLConnet("l6a01_project"))
print(len(df))
df.iloc[0,:].to_dict()

8880


{'line_id': 'CAPIC100',
 'aoi': 'aoi200',
 'model': 'B160UAN05',
 'recipe_id': '0258',
 'glass_type': 'TFT',
 'ai_code_1': 'PIS With Particle',
 'size_key': 'S',
 'total_glass_count': 7,
 'defect_count': 0.0,
 'density': 0.0,
 'overD': 0.0,
 'removed_glasses': 0,
 'removed_defects': 0.0,
 'final_glass_count': 7,
 'final_defect_count': 0.0,
 'final_density': 0.0,
 'std': 0.0,
 'OOC': 0.0,
 'OOS': 0.0,
 'GEN_DT': '251204095506'}

In [19]:
df.columns

Index(['line_id', 'aoi', 'model', 'recipe_id', 'glass_type', 'ai_code_1',
       'size_key', 'total_glass_count', 'defect_count', 'density', 'overD',
       'removed_glasses', 'removed_defects', 'final_glass_count',
       'final_defect_count', 'final_density', 'std', 'OOC', 'OOS', 'GEN_DT'],
      dtype='object')

In [7]:
tbn = 'aoi100_pidensity_202511_pi600'
tb= dbhandler.get_table(tbn)
tb.iloc[-1,:].to_dict()

{'scan_time': Timestamp('2025-11-29 22:32:26'),
 'line_id': 'CAPIC600',
 'model': 'M195RTN01',
 'glass_type': 'CF',
 'recipe_id': '803',
 'glass_id': 'AJ5R1B1XBC',
 'pic_name': 'CaptureImage/CAPIT203_AJ5R1B1XBC_9_46_2511292232.JPG',
 'x': '1363773',
 'y': '85564',
 'predict_code': '!',
 'judge_code': '!',
 'mark': '',
 'hour': '',
 'dayhour': '',
 'day': Timestamp('2025-11-29 22:32:26'),
 'year': '',
 'month': '',
 'season': '',
 'week': '',
 'yearmonth': '',
 'defect_count': '54',
 'defect_size': '12',
 'open_status': '',
 'ai_code_1': 'OP_Defect',
 'ai_code_2': '2025-11-29 17:19:17',
 'ai_code_3': '',
 'ai_code_4': '',
 'ai_code_5': '',
 'gray_name': 'CCDImage\\IP01\\0040',
 'ip_num': '',
 'first_code': '',
 'chip_name': 'AJ5R1B1XBCC',
 'defect_seq': '9',
 'pi_time': '2025-11-29 18:07:17',
 'pi_hour': '2025-11-29 18',
 'pic_path': 'http://10.97.139.98:1454//CAPIT203/CA0322/AJ5R1B1XBC/CAPIC700/20251129223226/',
 'recipe_comment': ''}

In [48]:
tbn = "aoi_density_fixed_spec_table"
df = dbhandler.get_table(tbn)
print(len(df))
df = df[df['GEN_DT'].str.contains('251209')]
print(len(df))
df.iloc[0,:].to_dict()

63540
12150


{'line_id': 'CAPIC100',
 'aoi': 'aoi200',
 'model': 'B160UAN03',
 'recipe_id': '0297',
 'glass_type': 'all',
 'ai_code_1': 'PIS With Particle',
 'size_key': 'M',
 'total_glass_count': 6,
 'defect_count': 0.0,
 'density': 0.0,
 'overD': 0.0,
 'removed_glasses': 0,
 'removed_defects': 0.0,
 'final_glass_count': 6,
 'final_defect_count': 0.0,
 'final_density': 0.0,
 'std': 0.0,
 'OOC': 0.0,
 'OOS': 0.0,
 'GEN_DT': '251209073050'}

In [51]:
match_dict = {'line_id': 'CAPIC200',
 'aoi': 'aoi200',
 'model': 'M315QAN01',
 'recipe_id': '2287',
 'glass_type': 'TFT',
 'ai_code_1': 'SSIU_Polymer',
 'size_key': 'SMLO'}

for cl, val in match_dict.items():
    df2 = df[df[cl] == val]
    print(cl, val, len(df2))
    if df2.empty:
        break
    else:
        df = df2.copy()
df


line_id CAPIC200 1980
aoi aoi200 1980
model M315QAN01 210
recipe_id 2287 150
glass_type TFT 75
ai_code_1 SSIU_Polymer 15
size_key SMLO 1


,line_id,aoi,model,recipe_id,glass_type,ai_code_1,size_key,total_glass_count,defect_count,density,overD,removed_glasses,removed_defects,final_glass_count,final_defect_count,final_density,std,OOC,OOS,GEN_DT
62821,CAPIC200,aoi200,M315QAN01,2287,TFT,SSIU_Polymer,SMLO,32,2677.0,83.65625,250.96875,0,0.0,32,2677.0,83.65625,36.24256,192.383931,301.111612,251209073050


In [3]:
# ===== Logging =====
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s [%(funcName)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)

In [3]:
class Config:
    def __init__(self):
        # ================================================= datetime =================================================
        now = datetime.now()
        ym = now.strftime("%Y%m")
        print(f"連線時間: {now}({ym})")
        one_mon_ago = now - timedelta(days=30)
        # ================================================= 資料設定  =================================================
        # === PI Line & AOI Tool  ====
        self.uni_aoi_names = [f'aoi{i}00' for i in range(1,4,1)]
        self.uni_pi_names = [f'pi{i}00' for i in range(1,8,1)]
        # === particle type & defect code ====
        self.uni_UPI_defect_codes = ['Polymer', 'SSIU_Polymer']
        self.uni_SPOT_defect_codes = ['PI_Spot_NP', 'PIS With Particle']
        self.uni_SPS_defect_codes= ['SPS']
        self.uni_defect_types = ['Particle', 'PISpot']
        self.all_defect_codes = ['Polymer', 'SSIU_Polymer', 'PI_Spot_NP', 'PIS With Particle', 'SPS']
        # ==== defect size config  ===
        self.uni_defect_sizes = ['S', 'M', 'L', 'O']
        self.rawdata_defect_size_col = 'defect_size'
        self.defect_size_rules = {
            "S": lambda x: x <= 20,
            "M": lambda x: 21 <= x <= 100,
            "L": lambda x: 101 <= x <= 400,
            "O": lambda x: x >= 401,
        }
        # === Glass side  ===
        self.glass_sides = ['CF', 'TFT']

        # =================================================  UPI/SPOT Default config  ================================================= 
        
        self.tab_filter_config = {
            'UPI': {
                'line_id': ['CAPIC200'],
                'ai_code_1': self.uni_UPI_defect_codes,
                'recipe_id':[]
                },
            'PISpot': {
                'line_id': ['CAPIC200'],
                'ai_code_1': self.uni_SPOT_defect_codes,
                'recipe_id':[]
                },
            'SPS': {
                'line_id': [f'CAPIC{i}00' for i in range(1,8,1)],
                'ai_code_1': self.uni_SPS_defect_codes,
                'recipe_id':[]
                },
            'SPEC':{

            }
        }
        
        # ================================================= MySQL AOI Density 資料表設定 ================================================= 
        self.aoi_pidensit_summary_tbn = f'pidensity_{ym}'
        self.aoi_pi_density_tbns = [f'{aoi}_pidensity_yyyymm_{pi}' for aoi in self.uni_aoi_names for pi in self.uni_pi_names]
        self.aoi_pidensity_spec_tbn = 'aoi_spec_for_aoimonitor'
        print('預設讀取:',self.aoi_pidensit_summary_tbn)


        self.aoi_pidensity_spec_cols = ['NO', 'MODEL_ID', 'MODEL_TYPE', 'DEFECT_CODE', 'PROCESS_TYPE', 'SIZE_TYPE', 'OOC', 'OOS']
        self.aoi_density_summary_sql_cols = ['pi_hour', 'aoi', 'pi', 'line_id', 'model', 'glass_type', 'recipe_id','defect_type', 'ai_code_1', 
                                             'n_rows', 'n_glasses', 'small_defect_count','middle_defect_count', 'large_defect_count', 'over_defect_count',
                                             'unknown_defect_count', 'glass']
        
        self.aoi_density_rawdata_sql_cols = ['scan_time', 'line_id', 'model', 'glass_type', 'recipe_id', 'glass_id',
       'pic_name', 'x', 'y', 'predict_code', 'judge_code', 'mark', 'hour','dayhour', 'day', 'year', 'month', 'season', 'week', 'yearmonth',
       'defect_count', 'defect_size', 'open_status', 'ai_code_1', 'ai_code_2','ai_code_3', 'ai_code_4', 'ai_code_5', 'gray_name', 'ip_num',
       'first_code', 'chip_name', 'defect_seq']
        
        self.sql_group_keys = ['line_id', 'aoi', 'model', 'glass_type', 'recipe_id','defect_type', 'ai_code_1','pi_hour'  ]
        
        # ================================================= 前端網頁CONFIG ================================================= 
        self.chart_table_coldict = {
                                    'line_id': 'PI Line',
                                    'aoi':'aoi',
                                    'model': 'Model', 
                                    'glass_type': 'side',
                                    'pi_hour': 'Hourly',
                                    'recipe_id':'recipe',
                                    'ai_code_1': 'defect', 
                                    'n_glasses': 'glassNum', 
                                    'n_rows': 'defNum',
                                    'glass': 'glass', #list ,group中的所有glass
                                    'glass_defect_count': 'glass def statis'} #dict, group中的所有glass_id對應單片glass defect
        """
        
        """   
                                                
        self.chart_group_dict = {
            'left':['line_id','model','n_glasses'],
            'up': [ 'aoi', 'ai_code_1'] ,
            'down':['pi_hour'],
            'right':['density'],
            }
        
        self.uni_glass_row_info_dict = {'glass_id': 'glass', 
                                        'glass_defect_count':'glass_defect_count',  
                                        'small_defect_count': 'S',
                                        'middle_defect_count': 'M', 
                                        'large_defect_count': 'L', 
                                        'over_defect_count': 'O'}
        self.defect_group_coldict = {
            'x': 'x',
            'y': 'y',
            'chip_name': 'chip',
            'pic_name':'img'}
        
        self.filter_item_coldict = {
            'line_id': 'PI Line',
            'aoi':'aoi tools',
            'model': 'Model', 
            'recipe_id':'recipe',
            'glass_type': 'glass_side',
            'ai_code_1': 'defect code', 
            'defect_size': 'defect size',
        }


        self.front_config = {
            'chartKeyDict': self.chart_group_dict,
            'filtetItemKeyDict':self.filter_item_coldict,
            'hourlyTable': self.chart_table_coldict,
            'uniGlassInfo': self.uni_glass_row_info_dict,
            'uniGlassDefectTable': self.defect_group_coldict,
            'SubTabsFilterDefaultDict':self.tab_filter_config
        }
# -------- 小工具 --------


In [ ]:
def months_in_last_n_days(ref: datetime, days: int = 90) -> List[str]:
    """
    取「近 N 天」內，所有第一天落在此區間內的月份（YYYYMM），用來粗篩表名中的 yyyymm。
    例如 ref=2025-11-04, days=90 -> ['202511','202510','202509']
    """
    threshold = ref - timedelta(days=days)
    cur = datetime(ref.year, ref.month, 1)
    out = []
    while cur >= datetime(threshold.year, threshold.month, 1):
        out.append(f"{cur.year}{cur.month:02d}")
        # 前一個月第一天
        if cur.month == 1:
            cur = datetime(cur.year - 1, 12, 1)
        else:
            cur = datetime(cur.year, cur.month - 1, 1)
    return out

def months_for_window(end_dt_tz: datetime, days: int = 90) -> list[str]:
    """
    取 [end_dt - days, end_dt) 視窗內所涵蓋的 YYYYMM（用月初做遞減）。
    end_dt_tz 必須是 timezone-aware。
    """
    start_dt_tz = end_dt_tz - timedelta(days=days)
    cur = end_dt_tz.replace(day=1, hour=0, minute=0, second=0, microsecond=0)
    out = []
    while cur >= start_dt_tz.replace(day=1, hour=0, minute=0, second=0, microsecond=0):
        out.append(f"{cur.year}{cur.month:02d}")
        if cur.month == 1:
            cur = cur.replace(year=cur.year-1, month=12)
        else:
            cur = cur.replace(month=cur.month-1)
    return out

def compute_cutoff_730_local(ref_datetime: datetime | None = None,
                             tz: str = "Asia/Taipei",
                             hh: int = 7, mm: int = 30) -> datetime:
    """
    回傳「當天 hh:mm:00」的 timezone-aware datetime（預設 07:30, Asia/Taipei）。
    若 ref_datetime 為 None，取現在時間；若 ref 是 naive，視為 tz 當地時間。
    """
    tzinfo = ZoneInfo(tz)
    if ref_datetime is None:
        now = datetime.now(tzinfo)
    else:
        now = ref_datetime
        if now.tzinfo is None:
            now = now.replace(tzinfo=tzinfo)
        else:
            now = now.astimezone(tzinfo)
    return now.replace(hour=hh, minute=mm, second=0, microsecond=0)

# ===== 表名 Regex =====
_PID_TBL_RE = re.compile(r'^(aoi[0-9]{3})_pidensity_([0-9]{6})_pi([0-9]{3})$')


# ===== 列表符合命名規則的 pidensity 資料表 =====
def list_pidensity_tables(conn: Connection, dbname: str, yyyymm_whitelist: Set[str],
                          aoi_filter: Optional[Iterable[str]] = None,
                          pi_filter: Optional[Iterable[str]] = None) -> List[str]:
    """
    從 information_schema.table 列出符合命名規則的表，再以 yyyymm / aoi / pi 篩選。
    """
    sql = text("""
        SELECT table_name
        FROM information_schema.tables
        WHERE table_schema = :db
          AND table_name REGEXP '^aoi[0-9]{3}_pidensity_[0-9]{6}_pi[0-9]{3}$'
    """)
    names = conn.execute(sql, {"db": dbname}).scalars().all()

    aoi_set = set(aoi_filter) if aoi_filter else None
    pi_set  = set(pi_filter) if pi_filter else None

    picked = []
    for t in names:
        m = _PID_TBL_RE.match(t)
        if not m:
            continue
        aoi_str, yyyymm, pi3 = m.groups()
        if yyyymm not in yyyymm_whitelist:
            continue
        if aoi_set and aoi_str not in aoi_set:
            continue
        if pi_set and pi3 not in pi_set:
            continue
        picked.append(t)
    logging.info(f"候選表數量：{len(picked)} / 原始 {len(names)}（已按月份與可選 aoi/pi 粗篩）")
    return sorted(picked)


# ===== 查詢表欄位（做容錯） =====
def get_table_columns(conn: Connection, dbname: str, table_name: str) -> Set[str]:
    sql = text("""
        SELECT column_name
        FROM information_schema.columns
        WHERE table_schema = :db AND table_name = :tbl
    """)
    return set(conn.execute(sql, {"db": dbname, "tbl": table_name}).scalars().all())


# ===== 從單表抓資料（SQL 端先過濾 scan_time + recipe_id） =====
def fetch_one_table(conn, dbname: str, table_name: str,
                    start_dt: datetime, end_dt: datetime,
                    need_cols: list[str]) -> pd.DataFrame:
    try:
        avail = get_table_columns(conn, dbname, table_name)
    except Exception as e:
        logging.warning(f"[{table_name}] 無法讀取欄位，略過。err={e}")
        return pd.DataFrame(columns=need_cols)

    if 'scan_time' not in avail:
        logging.info(f"[{table_name}] 無 scan_time 欄位，略過。")
        return pd.DataFrame(columns=need_cols)

    select_cols = [c for c in need_cols if c in avail and c not in ('aoi', 'size_bucket')]
    if not select_cols:
        logging.info(f"[{table_name}] 無可選欄位，略過。")
        return pd.DataFrame(columns=need_cols)

    # 注意：右界使用 < end_dt（嚴格早於 07:30）。若你想含 07:30，改成 <= :end_dt
    sql = text(f"""
        SELECT {', '.join([f'`{c}`' for c in select_cols])}
        FROM `{table_name}`
        WHERE `scan_time` >= :start_dt
          AND `scan_time` <  :end_dt
          AND (
            SUBSTRING(CAST(`recipe_id` AS CHAR), 1, 1) IN ('2','0')
            OR CHAR_LENGTH(CAST(`recipe_id` AS CHAR)) = 3
          )
    """)

    try:
        df = pd.read_sql_query(sql, conn, params={"start_dt": start_dt, "end_dt": end_dt})
    except Exception as e:
        logging.warning(f"[{table_name}] 查詢失敗，略過。err={e}")
        return pd.DataFrame(columns=need_cols)

    m = _PID_TBL_RE.match(table_name)
    df['aoi'] = (m.group(1) if m else None)

    for c in need_cols:
        if c not in df.columns:
            df[c] = pd.NA

    try:
        df['scan_time'] = pd.to_datetime(df['scan_time'], errors='coerce', utc=False)
    except Exception:
        pass

    if 'defect_size' in df.columns:
        df['defect_size'] = pd.to_numeric(df['defect_size'], errors='coerce')
        bins = [-float('inf'), 20, 100, 400, float('inf')]
        labels = ['S', 'M', 'L', 'O']
        df['size_bucket'] = pd.cut(df['defect_size'], bins=bins, labels=labels, right=True)
    else:
        df['size_bucket'] = pd.NA

    return df[need_cols]
GLS_GROUP_KEYS = ['line_id', 'aoi', 'model', 'recipe_id', 'glass_type', 'glass_id']

def keep_latest_measurement_group(
    df: pd.DataFrame,
    time_col: str = 'scan_time',
    key_cols = GLS_GROUP_KEYS,
    print_limit: int = 500
):
    """
    依 key_cols 分群，若同群內有多個 scan_time，僅保留最新 scan_time 的整批資料。
    會印出被刪除的 (群 + scan_time + 刪除筆數)；回傳 (filtered_df, removed_df)。
    """
    tb = df.copy()
    
    # 1) 時間欄位轉成 datetime（容錯）
    if time_col not in tb.columns:
        raise KeyError(f"缺少時間欄位: {time_col}")
    tb[time_col] = pd.to_datetime(tb[time_col], errors='coerce')

    # 2) 驗證 key 欄
    missing = [c for c in key_cols if c not in tb.columns]
    if missing:
        raise KeyError(f"缺少必要分群欄位: {missing}")

    # 3) 每群的最新時間（transform 回填到每列）
    grp_max = tb.groupby(key_cols, dropna=False)[time_col].transform('max')

    # 4) 標記要保留的列：時間=該群最大時間；若該群的最大時間為 NaT（全 NaT），則保留整群
    keep_mask = (tb[time_col] == grp_max) | grp_max.isna()

    kept = tb[keep_mask].copy()
    removed = tb[~keep_mask].copy()

    # 5) 印出被刪除的群與 scan_time
    if not removed.empty:
        summary = (
            removed.groupby(key_cols + [time_col], dropna=False)
                   .size().reset_index(name='rows_removed')
                   .sort_values(key_cols + [time_col])
        )
        # 僅印前 print_limit 筆，避免洗版
        for _, row in summary.head(print_limit).iterrows():
            grp_info = ", ".join(f"{k}={row[k]}" for k in key_cols)
            print(f"[REMOVED] {grp_info} | scan_time={row[time_col]} | rows={row['rows_removed']}")
        if len(summary) > print_limit:
            print(f"... and {len(summary) - print_limit} more removed groups.")
    else:
        print("沒有發現同群多次量測（無需刪除）。")
    kept.reset_index(drop=True, inplace = True)
    removed.reset_index(drop=True, inplace = True)

    print(f'確認前筆數:{len(df)}')
    print(f'確認後筆數:{len(kept)}')
    print(f'刪除筆數:{len(removed)}')
    return kept, removed



# ===== 主流程：抓多表合併 =====
def fetch_pidensity_recent(mysql: MySQLConnet,
                           days: int = 90,
                           ref_datetime: datetime | None = None,
                           aoi_filter: Iterable[str] | None = None,
                           pi_filter: Iterable[str] | None = None,
                           *,
                           tz: str = "Asia/Taipei",
                           cutoff_hh: int = 7, cutoff_mm: int = 30) -> pd.DataFrame:
    """
    以「當天 {cutoff_hh}:{cutoff_mm}（{tz}）」為截止點，回傳近 days 天資料（scan_time ∈ [start, end)）。
    """
    engine = mysql.engine
    dbname = mysql.db

    # 1) 計算截止點（含時區），與起始點；轉成「naive 本地時間」給 MySQL
    end_dt_tz = compute_cutoff_730_local(ref_datetime, tz=tz, hh=cutoff_hh, mm=cutoff_mm)
    start_dt_tz = end_dt_tz - timedelta(days=days)
    end_dt = end_dt_tz.replace(tzinfo=None)
    start_dt = start_dt_tz.replace(tzinfo=None)

    # 2) 以視窗涵蓋月份粗篩表名
    months = set(months_for_window(end_dt_tz, days=days))  # e.g., {'202507','202506','202505'}

    final_cols = ['scan_time', 'line_id', 'aoi', 'model', 'recipe_id',
                  'glass_type', 'glass_id', 'ai_code_1', 'defect_size', 'size_bucket']

    frames: list[pd.DataFrame] = []
    with engine.connect() as conn:
        tables = list_pidensity_tables(conn, dbname, months,
                                       aoi_filter=aoi_filter, pi_filter=pi_filter)
        logging.info(f"開始查詢 {len(tables)} 張表 ... "
                     f"起始：{start_dt:%Y-%m-%d %H:%M:%S}, 截止（不含）：{end_dt:%Y-%m-%d %H:%M:%S}")

        for tbl in tables:
            df = fetch_one_table(conn, dbname, tbl, start_dt, end_dt, final_cols)
            if not df.empty:
                frames.append(df)

    if not frames:
        logging.info("沒有符合條件的資料。")
        return pd.DataFrame(columns=final_cols)

    out = pd.concat(frames, ignore_index=True)
    if 'scan_time' in out.columns:
        out = out.sort_values('scan_time', ascending=False, kind='mergesort')
    return out.reset_index(drop=True)
# ===== 可直接執行 =====
if __name__ == "__main__":
    dbha = MySQLConnet("l6a01_project")
    cfg = Config()
    # 例：抓全部 AOI、全部 PI，預設 90 天（以現在時間為基準）
    df = fetch_pidensity_recent(mysql, days=90)
    print(f'資料庫撈取完成,筆數:{len(df)}')
    print(df.head(2))
    kept, removed = keep_latest_measurement_group(df)
    # 若要指定基準時間（如報表跑到某天止）：
    # ref_day = datetime(2025, 11, 4, 12, 0, 0)
    # df = fetch_pidensity_recent(mysql, days=90, ref_datetime=ref_day)

    # 若只抓特定 AOI 與部分 PI（例如 pi=000/100/200）：
    # df = fetch_pidensity_recent(
    #         mysql, days=90,
    #         aoi_filter=['aoi100','aoi200'],
    #         pi_filter=['000','100','200']
    #      )
    # print(df.tail(10)

2025-11-04 16:29:17,353 - INFO - 候選表數量：48 / 原始 48（已按月份與可選 aoi/pi 粗篩）
2025-11-04 16:29:17,354 - INFO - 開始查詢 48 張表 ... 起始：2025-08-06 07:30:00, 截止（不含）：2025-11-04 07:30:00


資料庫撈取完成,筆數:1336453
            scan_time   line_id     aoi      model recipe_id glass_type  \
0 2025-11-03 19:41:15  CAPIC300  aoi200  B140UAN04      2299        TFT   
1 2025-11-03 19:41:15  CAPIC300  aoi200  B140UAN04      2299        TFT   

    glass_id   ai_code_1  defect_size size_bucket  
0  JF5AASW1C   Not_Found            8           S  
1  JF5AASW1C  PI_Spot_NP           56           M  
[REMOVED] line_id=CAPIC100, aoi=aoi100, model=M215HAN01, recipe_id=117, glass_type=CF, glass_id=AL5HZA0BF0 | scan_time=2025-10-07 14:20:19 | rows=48
[REMOVED] line_id=CAPIC100, aoi=aoi200, model=B116XAN06, recipe_id=0289, glass_type=TFT, glass_id=A25A8WS6A | scan_time=2025-09-16 20:56:03 | rows=9
[REMOVED] line_id=CAPIC100, aoi=aoi200, model=B116XAN06, recipe_id=0289, glass_type=TFT, glass_id=A25A8WS6B | scan_time=2025-09-16 20:51:27 | rows=1
[REMOVED] line_id=CAPIC100, aoi=aoi200, model=B116XAN06, recipe_id=0289, glass_type=TFT, glass_id=A25A8WS6C | scan_time=2025-09-16 20:46:53 | rows=3
[RE

In [ ]:


# ===== 設定 =====
DEFAULT_GLASS_SIDES  = cfg.glass_sides  # 會另加 'all'
DEFAULT_DEFECT_CODES = cfg.all_defect_codes
DEFAULT_SIZE_KEYS    = cfg.size_group_keys

MAIN_KEYS     = ['line_id', 'aoi', 'model', 'recipe_id']
GLASS_ID_KEYS = ['scan_time', 'glass_id']
SIZE_ATOMS    = ['S','M','L','O']

def _size_combo_map(size_key_names: Sequence[str]) -> Dict[str, Tuple[str, ...]]:
    out = {}
    for k in size_key_names:
        out[k] = tuple([ch for ch in k if ch in SIZE_ATOMS])
    return out

def _cast_to_category(df: pd.DataFrame, cols: Sequence[str]):
    for c in cols:
        if c in df.columns and df[c].dtype == object:
            df[c] = df[c].astype('category')

def compute_dynamic_spec_summary_streaming(
    df: pd.DataFrame,
    glass_sides: Sequence[str] = DEFAULT_GLASS_SIDES,
    all_defect_codes: Sequence[str] = DEFAULT_DEFECT_CODES,
    size_key_names: Sequence[str] = DEFAULT_SIZE_KEYS,
    verbose: bool = False,
) -> pd.DataFrame:
    """
    低記憶體串流版：
      - 分母：全部片（到 glass_type 以前不看 code）；ALL=CF∪TFT 去重
      - 分子：僅重點 codes；逐群處理，不做全表 pivot
      - 標準差：用 Σx、Σx² 與 n 計算，隱含 0（沒缺陷的片）
    """
    need = set(MAIN_KEYS + ['glass_type','ai_code_1','size_bucket'] + GLASS_ID_KEYS)
    miss = need - set(df.columns)
    if miss:
        raise KeyError(f"缺少必要欄位: {sorted(miss)}")

    tb = df[MAIN_KEYS + ['glass_type','ai_code_1','size_bucket'] + GLASS_ID_KEYS].copy()
    tb['scan_time'] = pd.to_datetime(tb['scan_time'], errors='coerce')

    # 節省記憶體：高重複欄位轉 category
    _cast_to_category(tb, MAIN_KEYS + ['glass_type','ai_code_1','size_bucket'])

    # ===== 1) 分母：全部片（ALL=CF∪TFT 去重） =====
    side_whitelist = set(list(glass_sides) + ['all'])

    # CF/TFT 各自分母
    raw_glasses = tb[MAIN_KEYS + ['glass_type'] + GLASS_ID_KEYS].drop_duplicates()
    raw_tot = (raw_glasses
               .groupby(MAIN_KEYS + ['glass_type'], dropna=False, observed=True)
               .size())
    # ALL 分母（忽略 glass_type 後去重）
    all_glasses = tb[MAIN_KEYS + GLASS_ID_KEYS].drop_duplicates()
    all_tot = (all_glasses
               .groupby(MAIN_KEYS, dropna=False, observed=True)
               .size())

    # 建 tot_map 查表
    tot_map: Dict[Tuple, int] = {}
    for idx, v in raw_tot.items():
        *main, gtype = idx
        if gtype in side_whitelist:
            tot_map[tuple(main)+(''+gtype,)] = int(v)
    for idx, v in all_tot.items():
        tot_map[tuple(idx)+('all',)] = int(v)

    # ===== 2) 分子：僅重點 codes；逐群處理 =====
    combo_map = _size_combo_map(size_key_names)
    out_rows: List[dict] = []

    # 只留下重點 code 進行分子計算（不影響分母）
    tf = tb[tb['ai_code_1'].isin(all_defect_codes)].copy()
    _cast_to_category(tf, MAIN_KEYS + ['glass_type','ai_code_1','size_bucket'])

    # 先排序，讓 groupby 產生小塊迭代，減少記憶體峰值
    tf.sort_values(by=MAIN_KEYS + ['glass_type','ai_code_1','glass_id','scan_time'],
                   kind='mergesort', inplace=True)

    # === 2a) 製程分群（CF/TFT）：逐 (主群 + glass_type + code) 迭代
    for (line_id, aoi, model, recipe_id, glass_type, ai_code), g in \
        tf.groupby(MAIN_KEYS + ['glass_type','ai_code_1'], sort=False, observed=True):

        if glass_type not in side_whitelist or glass_type == 'all':
            continue  # 'all' 在 2b 處理

        tg = int(tot_map.get((line_id, aoi, model, recipe_id, glass_type), 0) or 0)
        if tg == 0:
            # 沒分母：直接輸出 NaN 一列 per size_key
            for size_key in size_key_names:
                out_rows.append({
                    'line_id': line_id, 'aoi': aoi, 'model': model, 'recipe_id': recipe_id,
                    'glass_type': glass_type, 'ai_code_1': ai_code, 'size_key': size_key,
                    'total_glass_count': 0, 'defect_count': 0.0, 'density': np.nan, 'overD': np.nan,
                    'removed_glasses': 0, 'removed_defects': 0.0,
                    'final_glass_count': np.nan, 'final_defect_count': 0.0, 'final_density': np.nan,
                    'std': np.nan, 'OOC': np.nan, 'OOS': np.nan
                })
            continue

        # 每片×尺寸原子計數（只用到 S/M/L/O）
        # 注意：這裡只在「這個小群」上 groupby，記憶體很省
        per_glass = (g[g['size_bucket'].isin(SIZE_ATOMS)]
                       .groupby(GLASS_ID_KEYS + ['size_bucket'], sort=False, observed=True)
                       .size()
                       .unstack('size_bucket', fill_value=0))
        if per_glass.empty:
            # 這個 (group, code) 完全沒有缺陷，分子=0，但標準差需要 n
            for size_key in size_key_names:
                final_tg = tg
                final_density = 0.0 if final_tg > 0 else np.nan
                std = np.nan if final_tg <= 1 else 0.0  # 全 0，變異 0
                OOC = (final_density + 3*std) if np.isfinite(final_density) and np.isfinite(std) else np.nan
                OOS = (final_density + 6*std) if np.isfinite(final_density) and np.isfinite(std) else np.nan
                out_rows.append({
                    'line_id': line_id, 'aoi': aoi, 'model': model, 'recipe_id': recipe_id,
                    'glass_type': glass_type, 'ai_code_1': ai_code, 'size_key': size_key,
                    'total_glass_count': tg, 'defect_count': 0.0, 'density': 0.0, 'overD': 0.0,
                    'removed_glasses': 0, 'removed_defects': 0.0,
                    'final_glass_count': final_tg, 'final_defect_count': 0.0, 'final_density': final_density,
                    'std': std, 'OOC': OOC, 'OOS': OOS
                })
            continue

        # 確保四個原子欄位存在
        for c in SIZE_ATOMS:
            if c not in per_glass.columns:
                per_glass[c] = 0
        # 逐尺寸組合
        for size_key, atoms in combo_map.items():
            x = per_glass[list(atoms)].sum(axis=1) if len(atoms) else pd.Series(0, index=per_glass.index)

            defect_count = float(x.sum())
            density = defect_count / tg if tg > 0 else np.nan
            overD = density * 3 if np.isfinite(density) else np.nan

            # outlier：只在有缺陷的片上判斷；0 不會被剔除
            removed_mask = (x > overD) if np.isfinite(overD) else pd.Series(False, index=x.index)
            removed_glasses = int(removed_mask.sum())
            removed_defects = float(x[removed_mask].sum())
            final_tg = tg - removed_glasses

            # 保留的片（仍然只包含「有缺陷的片」）
            xk = x[~removed_mask]
            sum_x = float(xk.sum())
            sum_x2 = float((xk**2).sum())
            final_defect_count = sum_x
            final_density = sum_x / final_tg if final_tg > 0 else np.nan

            if final_tg <= 1 or not np.isfinite(final_density):
                std = np.nan
            else:
                # 變異數：包含 0（沒缺陷）的片，透過 n 與 mean 隱含
                var = (sum_x2 - final_tg*(final_density**2)) / (final_tg - 1)
                std = np.sqrt(var) if np.isfinite(var) and var >= 0 else np.nan

            OOC = final_density + 3*std if np.isfinite(final_density) and np.isfinite(std) else np.nan
            OOS = final_density + 6*std if np.isfinite(final_density) and np.isfinite(std) else np.nan

            if verbose and removed_glasses > 0:
                head = x[removed_mask].head(5)
                print(f"[OUTLIER REMOVED][CF/TFT] line_id={line_id}, aoi={aoi}, model={model}, recipe_id={recipe_id}, "
                      f"glass_type={glass_type}, ai_code_1={ai_code}, size_key={size_key}, "
                      f"removed_glasses={removed_glasses}, removed_defects={removed_defects:.0f}, overD={overD:.6f}")
                if len(head) < removed_glasses:
                    print(f"... and {removed_glasses - len(head)} more")

            out_rows.append({
                'line_id': line_id, 'aoi': aoi, 'model': model, 'recipe_id': recipe_id,
                'glass_type': glass_type, 'ai_code_1': ai_code, 'size_key': size_key,
                'total_glass_count': tg, 'defect_count': defect_count, 'density': density, 'overD': overD,
                'removed_glasses': removed_glasses, 'removed_defects': removed_defects,
                'final_glass_count': final_tg, 'final_defect_count': final_defect_count, 'final_density': final_density,
                'std': std, 'OOC': OOC, 'OOS': OOS
            })

    # === 2b) ALL 不分製程：逐 (主群 + code) 迭代
    tf_all = tf.copy()
    tf_all = tf_all.drop(columns=['glass_type'])  # 忽略製程
    tf_all.sort_values(by=MAIN_KEYS + ['ai_code_1','glass_id','scan_time'],
                       kind='mergesort', inplace=True)

    for (line_id, aoi, model, recipe_id, ai_code), g in \
        tf_all.groupby(MAIN_KEYS + ['ai_code_1'], sort=False, observed=True):

        tg = int(tot_map.get((line_id, aoi, model, recipe_id, 'all'), 0) or 0)
        if tg == 0:
            for size_key in size_key_names:
                out_rows.append({
                    'line_id': line_id, 'aoi': aoi, 'model': model, 'recipe_id': recipe_id,
                    'glass_type': 'all', 'ai_code_1': ai_code, 'size_key': size_key,
                    'total_glass_count': 0, 'defect_count': 0.0, 'density': np.nan, 'overD': np.nan,
                    'removed_glasses': 0, 'removed_defects': 0.0,
                    'final_glass_count': np.nan, 'final_defect_count': 0.0, 'final_density': np.nan,
                    'std': np.nan, 'OOC': np.nan, 'OOS': np.nan
                })
            continue

        per_glass = (g[g['size_bucket'].isin(SIZE_ATOMS)]
                       .groupby(GLASS_ID_KEYS + ['size_bucket'], sort=False, observed=True)
                       .size()
                       .unstack('size_bucket', fill_value=0))
        if per_glass.empty:
            for size_key in size_key_names:
                final_tg = tg
                final_density = 0.0 if final_tg > 0 else np.nan
                std = np.nan if final_tg <= 1 else 0.0
                OOC = (final_density + 3*std) if np.isfinite(final_density) and np.isfinite(std) else np.nan
                OOS = (final_density + 6*std) if np.isfinite(final_density) and np.isfinite(std) else np.nan
                out_rows.append({
                    'line_id': line_id, 'aoi': aoi, 'model': model, 'recipe_id': recipe_id,
                    'glass_type': 'all', 'ai_code_1': ai_code, 'size_key': size_key,
                    'total_glass_count': tg, 'defect_count': 0.0, 'density': 0.0, 'overD': 0.0,
                    'removed_glasses': 0, 'removed_defects': 0.0,
                    'final_glass_count': final_tg, 'final_defect_count': 0.0, 'final_density': final_density,
                    'std': std, 'OOC': OOC, 'OOS': OOS
                })
            continue

        for c in SIZE_ATOMS:
            if c not in per_glass.columns:
                per_glass[c] = 0

        for size_key, atoms in combo_map.items():
            x = per_glass[list(atoms)].sum(axis=1) if len(atoms) else pd.Series(0, index=per_glass.index)

            defect_count = float(x.sum())
            density = defect_count / tg if tg > 0 else np.nan
            overD = density * 3 if np.isfinite(density) else np.nan

            removed_mask = (x > overD) if np.isfinite(overD) else pd.Series(False, index=x.index)
            removed_glasses = int(removed_mask.sum())
            removed_defects = float(x[removed_mask].sum())
            final_tg = tg - removed_glasses

            xk = x[~removed_mask]
            sum_x = float(xk.sum())
            sum_x2 = float((xk**2).sum())

            final_defect_count = sum_x
            final_density = sum_x / final_tg if final_tg > 0 else np.nan

            if final_tg <= 1 or not np.isfinite(final_density):
                std = np.nan
            else:
                var = (sum_x2 - final_tg*(final_density**2)) / (final_tg - 1)
                std = np.sqrt(var) if np.isfinite(var) and var >= 0 else np.nan

            OOC = final_density + 3*std if np.isfinite(final_density) and np.isfinite(std) else np.nan
            OOS = final_density + 6*std if np.isfinite(final_density) and np.isfinite(std) else np.nan

            if verbose and removed_glasses > 0:
                head = x[removed_mask].head(5)
                print(f"[OUTLIER REMOVED][ALL] line_id={line_id}, aoi={aoi}, model={model}, recipe_id={recipe_id}, "
                      f"ai_code_1={ai_code}, size_key={size_key}, "
                      f"removed_glasses={removed_glasses}, removed_defects={removed_defects:.0f}, overD={overD:.6f}")
                if len(head) < removed_glasses:
                    print(f"... and {removed_glasses - len(head)} more")
            row_dict = {
                'line_id': line_id, 'aoi': aoi, 'model': model, 'recipe_id': recipe_id,
                'glass_type': 'all', 'ai_code_1': ai_code, 'size_key': size_key,
                'total_glass_count': tg, 'defect_count': defect_count, 'density': density, 'overD': overD,
                'removed_glasses': removed_glasses, 'removed_defects': removed_defects,
                'final_glass_count': final_tg, 'final_defect_count': final_defect_count, 'final_density': final_density,
                'std': std, 'OOC': OOC, 'OOS': OOS
            }
            out_rows.append(row_dict)

    # ===== 3) 組裝輸出 =====
    cols = (MAIN_KEYS + ['glass_type','ai_code_1','size_key',
                         'total_glass_count','defect_count','density','overD',
                         'removed_glasses','removed_defects',
                         'final_glass_count','final_defect_count','final_density',
                         'std','OOC','OOS'])
    out = pd.DataFrame(out_rows, columns=cols) if out_rows else pd.DataFrame(columns=cols)

    # 排序輸出（依 size_key 順序）
    size_order = {k:i for i,k in enumerate(size_key_names)}
    out['__ord__'] = out['size_key'].map(size_order).fillna(9999).astype(int)
    out = out.sort_values(MAIN_KEYS + ['glass_type','ai_code_1','__ord__']).drop(columns='__ord__')
    out['GEN_DT'] = datetime.today().strftime('%y%m%d')
    return out.reset_index(drop=True)
result = compute_dynamic_spec_summary_streaming(df)


In [23]:
dbhandler.drop_table('his_90days_spec_table')

2025-11-04 16:59:21,852 - INFO - [drop_table] 資料表 'his_90days_spec_table' 已刪除.


In [ ]:
def filter_flexible(df, kv: dict, *, na_equal: bool=False, casefold: bool=False):
    """
    kv 的 value 可以是：
      - 單一值（含字串）→ 以等號比對
      - list/tuple/set → 以 isin 比對
    參數：
      na_equal=True  時，允許把 None/NaN 視為相等（單值或清單均適用）
      casefold=True  時，字串欄位與字串值都做大小寫無關比對（含 isin 的清單字串）
    """
    m = pd.Series(True, index=df.index)

    for col, val in kv.items():
        s = df[col]

        # 大小寫無關：把字串欄位與字串值（或清單中的字串）都做 casefold
        if casefold and is_string_dtype(s):
            s_cmp = s.astype("string").str.casefold()
            if isinstance(val, (list, tuple, set)):
                vals = [v.casefold() if isinstance(v, str) else v for v in val]
            else:
                vals = val.casefold() if isinstance(val, str) else val
        else:
            s_cmp = s
            vals = list(val) if isinstance(val, (list, tuple, set)) else val

        if isinstance(vals, list):  # isin 路徑
            # 先把非 NA 目標值拿來做 isin
            non_na_targets = []
            has_na_target = False
            for v in vals:
                if na_equal and (v is None or (isinstance(v, float) and math.isnan(v))):
                    has_na_target = True
                else:
                    non_na_targets.append(v)

            mask = s_cmp.isin(non_na_targets) if non_na_targets else pd.Series(False, index=s.index)
            if na_equal and has_na_target:
                mask = mask | s.isna()  # 用原欄位判斷 NaN
            m &= mask

        else:  # 等號路徑
            if na_equal and (vals is None or (isinstance(vals, float) and math.isnan(vals))):
                m &= s.isna()
            else:
                m &= s_cmp.eq(vals)

    return df.loc[m]

In [ ]:
import pandas as pd

SIZE_ATOMS = ['S','M','L','O']
GLASS_ID_KEYS = ['scan_time', 'glass_id']  

def atoms_from_size_key(size_key: str):
    """'SMLO' -> ['S','M','L','O']；只取 S/M/L/O 字母"""
    if size_key is None:
        return []
    return [ch for ch in str(size_key).upper() if ch in SIZE_ATOMS]

def _verify_one_group(product_rows: pd.DataFrame, result_rows: pd.DataFrame, main_key_tuple):
    """
    product_rows: 原 df 的某個主群資料（可含或不含 glass_type）;
    result_rows:  result 中對應同一主群（含 glass_type='all' 或 'CF'/'TFT'）的資料。
    """
    # 這裡只核對「defect_count」是否等於原 df 以 size 組合聚合後的筆數
    if result_rows.empty:
        return

    # 顯示該群分母（供參考）
    total_glass_count = product_rows.groupby(GLASS_ID_KEYS).ngroups
    print(f"[GROUP] {main_key_tuple} | total_glass_count={total_glass_count}, total_defects={len(product_rows)}")

    # 逐 (ai_code_1, size_key) 驗證
    for (ai_code, size_key), rsub in result_rows.groupby(['ai_code_1','size_key']):
        # result 每組應該只有一列
        row = rsub.iloc[0]
        expected_defects = int(row['defect_count'])

        atoms = atoms_from_size_key(size_key)
        if not atoms:
            # 空組合就當 0
            real_defects = 0
        else:
            # 在原 df（product_rows）中：ai_code_1 等於、size_bucket 落在該組合原子
            mask = (product_rows['ai_code_1'] == ai_code) & (product_rows['size_bucket'].isin(atoms))
            real_defects = int(mask.sum())

        if real_defects != expected_defects:
            print("  [MISMATCH]",
                  f"ai_code={ai_code}, size_key={size_key}, atoms={atoms}",
                  f"=> result={expected_defects}, df_count={real_defects}")
            # 如需看差異樣本，取消下行註解（只列前 5 筆）
            # print(product_rows.loc[mask, GLASS_ID_KEYS + ['size_bucket','ai_code_1']].head(5).to_string(index=False))

def verify_result_against_df(df: pd.DataFrame, result: pd.DataFrame):
    main_cols_with_side = ['line_id','aoi','model','recipe_id','glass_type']
    main_cols_no_side   = ['line_id','aoi','model','recipe_id']

    # 1) 驗證 CF/TFT（逐群）
    for main_key, product_rows in df.groupby(main_cols_with_side, dropna=False):
        # result 該群（CF 或 TFT）
        cond = {k:v for k,v in zip(main_cols_with_side, main_key)}
        result_rows = filter_flexible(result, cond)
        _verify_one_group(product_rows, result_rows, main_key)

    # 2) 驗證 ALL（不分製程 = CF ∪ TFT）
    for base_key, g in df.groupby(main_cols_no_side, dropna=False):
        # 把 CF/TFT 合在一起驗證
        cond_all = {**{k:v for k,v in zip(main_cols_no_side, base_key)}, 'glass_type': 'all'}
        result_rows_all = filter_flexible(result, cond_all)
        if result_rows_all.empty:
            continue
        _verify_one_group(g, result_rows_all, base_key + ('all',))

# ===== 執行驗證 =====
result = compute_dynamic_spec_summary_streaming(df)  # 你的輸出
verify_result_against_df(df, result)

[GROUP] ('CAPIC100', 'aoi100', 'B116XAN06', '815', 'CF') | total_glass_count=82, total_defects=7963
[GROUP] ('CAPIC100', 'aoi100', 'M215HAN01', '117', 'CF') | total_glass_count=86, total_defects=3425
[GROUP] ('CAPIC100', 'aoi100', 'M270HAN1C', '105', 'CF') | total_glass_count=15, total_defects=489
[GROUP] ('CAPIC100', 'aoi200', 'B116XAN06', '0289', 'TFT') | total_glass_count=139, total_defects=591
[GROUP] ('CAPIC100', 'aoi200', 'B116XAN06', '2289', 'TFT') | total_glass_count=140, total_defects=13097
[GROUP] ('CAPIC100', 'aoi200', 'B140UAN08', '0264', 'TFT') | total_glass_count=171, total_defects=10206
[GROUP] ('CAPIC100', 'aoi200', 'B140UAN08', '2264', 'TFT') | total_glass_count=171, total_defects=2951
[GROUP] ('CAPIC100', 'aoi200', 'B156HAN2U', '0309', 'TFT') | total_glass_count=551, total_defects=12575
[GROUP] ('CAPIC100', 'aoi200', 'B156HAN2U', '2309', 'TFT') | total_glass_count=558, total_defects=47799
[GROUP] ('CAPIC100', 'aoi200', 'B160UAN05', '0258', 'TFT') | total_glass_count=2

In [ ]:
"""
主分群: ['line_id', 'aoi', 'model', 'recipe_id']
    →目的: 機台及產品分群

第一層分群+組合:'glass_type' → cfg.glass_sides = ['CF', 'TFT'] + 'all' 
    →目的: 製成分群及總製成分群方便後續觀察趨勢
           取得同群內總片數(['scan_time','glass_id']相同算同一片上的資料)及總defect數量,總片數為density分母
    →製程群單一群參數:main_glass_type_rows 
    →群分母: total_glass_count = len(main_glass_type_rows.groupby(['scan_time','glass_id']))
子分群['ai_code_1']→ cfg.all_defect_codes = ['Polymer', 'SSIU_Polymer', 'PI_Spot_NP', 'PIS With Particle', 'SPS']
    →目的: 每個群僅保留all_defect_codes特定重點觀察之Defect code項目供後續計算

第二層分群+組合分群:['size_bucket'] → size_key_names = ['S','M','L','O','SM','SL','SO','ML','MO','LO','SML','SMO','SLO','MLO','SMLO']
    →目的: 計算每個群對應不同尺寸分群或組合分群的density
    →要避免組合時有可能重複計算導致錯誤結果

計算公式
最底層分群組合的單一群參數: size_group_data
size_group_data_defect_count = len(size_rows)
    → defect_code+特定尺寸組合之density: defect_code_size_density = size_group_data_defect_count / total_glass_count
    → 去除異常值標準: overD = defect_code_size_density * 3
size_group_data的['scan_time','glass_id']分群 →目的: 計算size_group_data中每片對應多少defect(size_glass_defect_count)
    驗證&去除異常值
    → size_group_data去除 size_glass_defect_count>overD的資料
        → 重新計算 size_group_data_defect_count = len(size_rows)  ###fixed
    → 該群(size_group)之total_glass_count需減去相對應片數
        → 重新計算 defect_code_size_density = size_group_data_defect_count / total_glass_count  ###fixed
取得樣本差
    std = (((size_glass_defect_count1 - defect_code_size_density)**2 + (size_glass_defect_count2 - defect_code_size_density)**2 ...)/ (total_glass_count - 1)) ** 0.5
計算NG標準:
OOC(WR)  = defect_code_size_density + std * 3
OOS(NG)  = defect_code_size_density + std * 6
"""




In [16]:
cfg.all_defect_codes



['Polymer', 'SSIU_Polymer', 'PI_Spot_NP', 'PIS With Particle', 'SPS']